In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

In [3]:
mon = pd.read_csv(r"C:\Users\magan\Downloads\MachineLearningCSV\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv")
tue = pd.read_csv(r"C:\Users\magan\Downloads\MachineLearningCSV\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv")

print(mon.head())
tue.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,88,640,7,4,440,358,220,0,62.857143,107.349008,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,88,900,9,4,600,2944,300,0,66.666667,132.287566,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1205,7,4,2776,2830,1388,0,396.571429,677.274651,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,88,511,7,4,452,370,226,0,64.571429,110.276708,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,773,9,4,612,2944,306,0,68.000000,134.933317,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [10]:
fri = pd.read_csv(r"C:\Users\magan\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")

In [8]:
mon[' Label'].value_counts()

 Label
BENIGN    529918
Name: count, dtype: int64

In [9]:
tue[' Label'].value_counts()

 Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

In [11]:
fri[' Label'].value_counts()

 Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

In [12]:
for i in mon.columns:
    print(i)

 Destination Port
 Flow Duration
 Total Fwd Packets
 Total Backward Packets
Total Length of Fwd Packets
 Total Length of Bwd Packets
 Fwd Packet Length Max
 Fwd Packet Length Min
 Fwd Packet Length Mean
 Fwd Packet Length Std
Bwd Packet Length Max
 Bwd Packet Length Min
 Bwd Packet Length Mean
 Bwd Packet Length Std
Flow Bytes/s
 Flow Packets/s
 Flow IAT Mean
 Flow IAT Std
 Flow IAT Max
 Flow IAT Min
Fwd IAT Total
 Fwd IAT Mean
 Fwd IAT Std
 Fwd IAT Max
 Fwd IAT Min
Bwd IAT Total
 Bwd IAT Mean
 Bwd IAT Std
 Bwd IAT Max
 Bwd IAT Min
Fwd PSH Flags
 Bwd PSH Flags
 Fwd URG Flags
 Bwd URG Flags
 Fwd Header Length
 Bwd Header Length
Fwd Packets/s
 Bwd Packets/s
 Min Packet Length
 Max Packet Length
 Packet Length Mean
 Packet Length Std
 Packet Length Variance
FIN Flag Count
 SYN Flag Count
 RST Flag Count
 PSH Flag Count
 ACK Flag Count
 URG Flag Count
 CWE Flag Count
 ECE Flag Count
 Down/Up Ratio
 Average Packet Size
 Avg Fwd Segment Size
 Avg Bwd Segment Size
 Fwd Header Length.1
Fwd Avg

In [13]:
df = pd.concat([tue, fri], axis=0)

print(df.head())

    Destination Port   Flow Duration   Total Fwd Packets  \
0                 88             640                   7   
1                 88             900                   9   
2                 88            1205                   7   
3                 88             511                   7   
4                 88             773                   9   

    Total Backward Packets  Total Length of Fwd Packets  \
0                        4                          440   
1                        4                          600   
2                        4                         2776   
3                        4                          452   
4                        4                          612   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                           358                     220   
1                          2944                     300   
2                          2830                    1388   
3                           370                 

In [14]:
print(df[' Label'].value_counts())

 Label
BENIGN         559611
PortScan       158930
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64


In [15]:
df.describe()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,732376.000000,7.323760e+05,732376.000000,732376.000000,7.323760e+05,7.323760e+05,732376.000000,732376.000000,732376.000000,732376.000000,...,732376.000000,7.323760e+05,7.323760e+05,7.323760e+05,7.323760e+05,7.323760e+05,7.323760e+05,7.323760e+05,7.323760e+05,7.323760e+05
mean,8552.354577,8.667200e+06,8.543314,9.844476,4.141169e+02,1.437042e+04,137.238558,16.579336,38.988481,39.998678,...,1.947636,-8.036352e+03,5.757406e+04,3.741335e+04,1.304328e+05,3.789129e+04,2.632839e+06,1.150914e+05,2.718669e+06,2.516003e+06
std,18020.315457,2.695970e+07,676.689970,914.810733,4.582997e+03,2.049129e+06,424.208430,34.319659,101.715442,136.515276,...,10.545025,2.080636e+06,5.654528e+05,3.256071e+05,9.400922e+05,5.140570e+05,1.125348e+07,1.724473e+06,1.155120e+07,1.109232e+07
min,0.000000,-1.300000e+01,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,53.000000,5.800000e+01,1.000000,1.000000,0.000000e+00,6.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,443.000000,3.910000e+02,2.000000,1.000000,4.900000e+01,7.200000e+01,33.000000,0.000000,29.000000,0.000000,...,0.000000,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,3534.250000,1.273688e+05,3.000000,2.000000,9.600000e+01,2.420000e+02,51.000000,37.000000,46.000000,0.377964,...,1.000000,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,65505.000000,1.200000e+08,206446.000000,276072.000000,2.428415e+06,6.270000e+08,24820.000000,2065.000000,4672.000000,5398.072094,...,2056.000000,1.380000e+02,1.100000e+08,7.050000e+07,1.100000e+08,1.100000e+08,1.200000e+08,7.590000e+07,1.200000e+08,1.200000e+08


In [21]:
df.columns = df.columns.str.strip()

In [23]:
zero_features=["Bwd PSH Flags","Fwd URG Flags","Bwd URG Flags","CWE Flag Count","Fwd Avg Bytes/Bulk","Fwd Avg Packets/Bulk",
               "Fwd Avg Bulk Rate","Bwd Avg Bytes/Bulk","Bwd Avg Packets/Bulk","Bwd Avg Bulk Rate"]


for i in zero_features:
    print(df[i].value_counts())

Bwd PSH Flags
0    732376
Name: count, dtype: int64
Fwd URG Flags
0    732376
Name: count, dtype: int64
Bwd URG Flags
0    732376
Name: count, dtype: int64
CWE Flag Count
0    732376
Name: count, dtype: int64
Fwd Avg Bytes/Bulk
0    732376
Name: count, dtype: int64
Fwd Avg Packets/Bulk
0    732376
Name: count, dtype: int64
Fwd Avg Bulk Rate
0    732376
Name: count, dtype: int64
Bwd Avg Bytes/Bulk
0    732376
Name: count, dtype: int64
Bwd Avg Packets/Bulk
0    732376
Name: count, dtype: int64
Bwd Avg Bulk Rate
0    732376
Name: count, dtype: int64


In [25]:
#drop fwd header.1 feature as it is repeated
drop_cols = zero_features +["Fwd Header Length.1"]

df1 = df.drop(drop_cols,axis=1)

len(df1.columns)

68

In [26]:
len(drop_cols)

11

In [114]:
#split 90% of each class present in the training
X = df1.drop('Label', axis=1)
y = df1['Label']

In [115]:
le = LabelEncoder()
y = le.fit_transform(y)

In [116]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)

{'BENIGN': 0, 'FTP-Patator': 1, 'PortScan': 2, 'SSH-Patator': 3}


In [35]:
print(np.isinf(X).sum().sum())
print(np.isnan(X).sum().sum()) 

1054
216


In [37]:
X = X.replace([np.inf, -np.inf], np.nan)

In [38]:
X = X.fillna(0)

In [39]:
print(np.isinf(X).sum().sum())
print(np.isnan(X).sum().sum())

0
0


In [40]:
# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,      # 10% test
    stratify=y,         # preserves label proportions
    random_state=42
)

In [41]:
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(659138, 67) (73238, 67) (659138,) (73238,)


## Training the model

In [33]:
xg = xgb.XGBClassifier(
    objective='multi:softprob',   # multiclass classification
    num_class=len(le.classes_),

    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method='hist',   # faster exact try with exact tmro night
    random_state=42,

    eval_metric='mlogloss'
)

In [42]:
xg.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None, num_class=4, ...)

## Testing the accuracy on the test set and live captures

In [43]:
y_pred = xg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

Accuracy: 0.9999590376580464

Classification Report:

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     55961
 FTP-Patator       1.00      1.00      1.00       794
    PortScan       1.00      1.00      1.00     15893
 SSH-Patator       1.00      1.00      1.00       590

    accuracy                           1.00     73238
   macro avg       1.00      1.00      1.00     73238
weighted avg       1.00      1.00      1.00     73238



* Model seems prettty accurate on the test set

## Testing on live capture

In [44]:
live1 = pd.read_csv(r"C:\Users\magan\Downloads\HPE Project\label_realdata_flows.csv")
live2 = pd.read_csv(r"C:\Users\magan\Downloads\HPE Project\port_brute_labeled.csv")

print(live1.head())
print(live2.head())

                                Flow ID     Src IP  Src Port          Dst IP  \
0  10.0.3.15-151.101.209.91-53042-443-6  10.0.3.15     53042  151.101.209.91   
1  10.0.3.15-151.101.209.91-53036-443-6  10.0.3.15     53036  151.101.209.91   
2  10.0.3.15-151.101.209.91-53028-443-6  10.0.3.15     53028  151.101.209.91   
3  10.0.3.15-151.101.209.91-53002-443-6  10.0.3.15     53002  151.101.209.91   
4  10.0.3.15-34.160.144.191-60774-443-6  10.0.3.15     60774  34.160.144.191   

   Dst Port  Protocol               Timestamp  Flow Duration  \
0       443         6  26/03/2026 10:17:45 PM         223089   
1       443         6  26/03/2026 10:17:45 PM         224891   
2       443         6  26/03/2026 10:17:45 PM         225495   
3       443         6  26/03/2026 10:17:45 PM         438803   
4       443         6  26/03/2026 10:17:46 PM          42589   

   Total Fwd Packet  Total Bwd packets  ...  Fwd Seg Size Min  Active Mean  \
0                 9                  9  ...             

In [45]:
print(len(live1.columns))
print(len(live2.columns))

84
84


In [53]:
for i in live1.columns:
    print(i)

Flow ID
Src IP
Src Port
Dst IP
Dst Port
Protocol
Timestamp
Flow Duration
Total Fwd Packet
Total Bwd packets
Total Length of Fwd Packet
Total Length of Bwd Packet
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Bytes/s
Flow Packets/s
Flow IAT Mean
Flow IAT Std
Flow IAT Max
Flow IAT Min
Fwd IAT Total
Fwd IAT Mean
Fwd IAT Std
Fwd IAT Max
Fwd IAT Min
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Bwd IAT Min
Fwd PSH Flags
Bwd PSH Flags
Fwd URG Flags
Bwd URG Flags
Fwd Header Length
Bwd Header Length
Fwd Packets/s
Bwd Packets/s
Packet Length Min
Packet Length Max
Packet Length Mean
Packet Length Std
Packet Length Variance
FIN Flag Count
SYN Flag Count
RST Flag Count
PSH Flag Count
ACK Flag Count
URG Flag Count
CWR Flag Count
ECE Flag Count
Down/Up Ratio
Average Packet Size
Fwd Segment Size Avg
Bwd Segment Size Avg
Fwd Bytes/Bulk Avg
Fwd Packet/Bulk Avg
Fw

In [ ]:
notin = []

for i in live1.columns:
    if (i not in df.columns):
        notin.append(i)
        
print(notin)

['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Packet Length Min', 'Packet Length Max', 'CWR Flag Count', 'Fwd Segment Size Avg', 'Bwd Segment Size Avg', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg', 'Bwd Bytes/Bulk Avg', 'Bwd Packet/Bulk Avg', 'Bwd Bulk Rate Avg', 'FWD Init Win Bytes', 'Bwd Init Win Bytes', 'Fwd Act Data Pkts', 'Fwd Seg Size Min']


In [91]:
# =========================================================
# RENAME MAP
# =========================================================

rename_map = {

    # Basic flow info
    'Dst Port': ' Destination Port',
    'Flow Duration': ' Flow Duration',

    # Packet counts
    'Total Fwd Packet': ' Total Fwd Packets',
    'Total Bwd packets': ' Total Backward Packets',

    # Packet lengths
    'Total Length of Fwd Packet': 'Total Length of Fwd Packets',
    'Total Length of Bwd Packet': ' Total Length of Bwd Packets',

    'Fwd Packet Length Max': ' Fwd Packet Length Max',
    'Fwd Packet Length Min': ' Fwd Packet Length Min',
    'Fwd Packet Length Mean': ' Fwd Packet Length Mean',
    'Fwd Packet Length Std': ' Fwd Packet Length Std',

    'Bwd Packet Length Max': 'Bwd Packet Length Max',
    'Bwd Packet Length Min': ' Bwd Packet Length Min',
    'Bwd Packet Length Mean': ' Bwd Packet Length Mean',
    'Bwd Packet Length Std': ' Bwd Packet Length Std',

    # Flow metrics
    'Flow Bytes/s': 'Flow Bytes/s',
    'Flow Packets/s': ' Flow Packets/s',

    # Flow IAT
    'Flow IAT Mean': ' Flow IAT Mean',
    'Flow IAT Std': ' Flow IAT Std',
    'Flow IAT Max': ' Flow IAT Max',
    'Flow IAT Min': ' Flow IAT Min',

    # Fwd IAT
    'Fwd IAT Total': 'Fwd IAT Total',
    'Fwd IAT Mean': ' Fwd IAT Mean',
    'Fwd IAT Std': ' Fwd IAT Std',
    'Fwd IAT Max': ' Fwd IAT Max',
    'Fwd IAT Min': ' Fwd IAT Min',

    # Bwd IAT
    'Bwd IAT Total': 'Bwd IAT Total',
    'Bwd IAT Mean': ' Bwd IAT Mean',
    'Bwd IAT Std': ' Bwd IAT Std',
    'Bwd IAT Max': ' Bwd IAT Max',
    'Bwd IAT Min': ' Bwd IAT Min',

    # Flags
    'Fwd PSH Flags': 'Fwd PSH Flags',
    'Bwd PSH Flags': ' Bwd PSH Flags',

    'Fwd URG Flags': ' Fwd URG Flags',
    'Bwd URG Flags': ' Bwd URG Flags',

    # Header lengths
    'Fwd Header Length': ' Fwd Header Length',
    'Bwd Header Length': ' Bwd Header Length',

    # Packets/sec
    'Fwd Packets/s': 'Fwd Packets/s',
    'Bwd Packets/s': ' Bwd Packets/s',

    # Packet statistics
    'Packet Length Min': ' Min Packet Length',
    'Packet Length Max': ' Max Packet Length',
    'Packet Length Mean': ' Packet Length Mean',
    'Packet Length Std': ' Packet Length Std',
    'Packet Length Variance': ' Packet Length Variance',

    # TCP flags
    'FIN Flag Count': 'FIN Flag Count',
    'SYN Flag Count': ' SYN Flag Count',
    'RST Flag Count': ' RST Flag Count',
    'PSH Flag Count': ' PSH Flag Count',
    'ACK Flag Count': ' ACK Flag Count',
    'URG Flag Count': ' URG Flag Count',
    'CWR Flag Count': ' CWE Flag Count',
    'ECE Flag Count': ' ECE Flag Count',

    # Ratios
    'Down/Up Ratio': ' Down/Up Ratio',

    # Average sizes
    'Average Packet Size': ' Average Packet Size',
    'Fwd Segment Size Avg': ' Avg Fwd Segment Size',
    'Bwd Segment Size Avg': ' Avg Bwd Segment Size',

    # Bulk
    'Fwd Bytes/Bulk Avg': 'Fwd Avg Bytes/Bulk',
    'Fwd Packet/Bulk Avg': ' Fwd Avg Packets/Bulk',
    'Fwd Bulk Rate Avg': ' Fwd Avg Bulk Rate',

    'Bwd Bytes/Bulk Avg': ' Bwd Avg Bytes/Bulk',
    'Bwd Packet/Bulk Avg': ' Bwd Avg Packets/Bulk',
    'Bwd Bulk Rate Avg': 'Bwd Avg Bulk Rate',

    # Subflow
    'Subflow Fwd Packets': 'Subflow Fwd Packets',
    'Subflow Fwd Bytes': ' Subflow Fwd Bytes',
    'Subflow Bwd Packets': ' Subflow Bwd Packets',
    'Subflow Bwd Bytes': ' Subflow Bwd Bytes',

    # Window sizes
    'FWD Init Win Bytes': 'Init_Win_bytes_forward',
    'Bwd Init Win Bytes': ' Init_Win_bytes_backward',

    # Active data
    'Fwd Act Data Pkts': ' act_data_pkt_fwd',
    'Fwd Seg Size Min': ' min_seg_size_forward',

    # Active/Idle
    'Active Mean': 'Active Mean',
    'Active Std': ' Active Std',
    'Active Max': ' Active Max',
    'Active Min': ' Active Min',

    'Idle Mean': 'Idle Mean',
    'Idle Std': ' Idle Std',
    'Idle Max': ' Idle Max',
    'Idle Min': ' Idle Min',

    # Label
    'Label': 'Label'
}

# =========================================================
# RENAME COLUMNS
# =========================================================

live1.rename(columns=rename_map, inplace=True)
live2.rename(columns=rename_map,inplace =True)
# =========================================================
# ORIGINAL REQUIRED FEATURES
# =========================================================

required_columns = [
    ' Destination Port',
    ' Flow Duration',
    ' Total Fwd Packets',
    ' Total Backward Packets',
    'Total Length of Fwd Packets',
    ' Total Length of Bwd Packets',
    ' Fwd Packet Length Max',
    ' Fwd Packet Length Min',
    ' Fwd Packet Length Mean',
    ' Fwd Packet Length Std',
    'Bwd Packet Length Max',
    ' Bwd Packet Length Min',
    ' Bwd Packet Length Mean',
    ' Bwd Packet Length Std',
    'Flow Bytes/s',
    ' Flow Packets/s',
    ' Flow IAT Mean',
    ' Flow IAT Std',
    ' Flow IAT Max',
    ' Flow IAT Min',
    'Fwd IAT Total',
    ' Fwd IAT Mean',
    ' Fwd IAT Std',
    ' Fwd IAT Max',
    ' Fwd IAT Min',
    'Bwd IAT Total',
    ' Bwd IAT Mean',
    ' Bwd IAT Std',
    ' Bwd IAT Max',
    ' Bwd IAT Min',
    'Fwd PSH Flags',
    ' Bwd PSH Flags',
    ' Fwd URG Flags',
    ' Bwd URG Flags',
    ' Fwd Header Length',
    ' Bwd Header Length',
    'Fwd Packets/s',
    ' Bwd Packets/s',
    ' Min Packet Length',
    ' Max Packet Length',
    ' Packet Length Mean',
    ' Packet Length Std',
    ' Packet Length Variance',
    'FIN Flag Count',
    ' SYN Flag Count',
    ' RST Flag Count',
    ' PSH Flag Count',
    ' ACK Flag Count',
    ' URG Flag Count',
    ' CWE Flag Count',
    ' ECE Flag Count',
    ' Down/Up Ratio',
    ' Average Packet Size',
    ' Avg Fwd Segment Size',
    ' Avg Bwd Segment Size',
    'Fwd Avg Bytes/Bulk',
    ' Fwd Avg Packets/Bulk',
    ' Fwd Avg Bulk Rate',
    ' Bwd Avg Bytes/Bulk',
    ' Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate',
    'Subflow Fwd Packets',
    ' Subflow Fwd Bytes',
    ' Subflow Bwd Packets',
    ' Subflow Bwd Bytes',
    'Init_Win_bytes_forward',
    ' Init_Win_bytes_backward',
    ' act_data_pkt_fwd',
    ' min_seg_size_forward',
    'Active Mean',
    ' Active Std',
    ' Active Max',
    ' Active Min',
    'Idle Mean',
    ' Idle Std',
    ' Idle Max',
    ' Idle Min',
    'Label'
]

In [92]:
live1.columns = live1.columns.str.strip()
live2.columns = live2.columns.str.strip()
print(live1.columns)
print(live2.columns)

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Destination Port',
       'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet L

In [93]:
# =========================================================
# DROP EXTRA COLUMNS
# =========================================================

drop_features = zero_features + ['Flow ID','Src IP','Src Port','Dst IP']

live1d = live1.drop(drop_features,axis=1)
live2d = live2.drop(drop_features,axis=1)

In [94]:
live1d.shape

(25709, 70)

In [95]:
live2d.shape

(83897, 70)

In [96]:
required_columns = [s.strip() for s in required_columns]

In [97]:
drop_cols = [col for col in live1d.columns if col not in required_columns]

print("Dropping columns:")
print(drop_cols)

live1d.drop(columns=drop_cols,inplace=True)
live2d.drop(columns=drop_cols,inplace=True)

Dropping columns:
['Protocol', 'Timestamp']


In [98]:
live2d.Label.value_counts()

Label
PortScan          75700
BENIGN             7633
FTP-BruteForce      454
SSH-BruteForce      110
Name: count, dtype: int64

In [121]:
live1d['Label'] = live1d['Label'].replace('BruteForce', 'SSH-Patator')
mapping = {'SSH-BruteForce': 'SSH-Patator', 'FTP-BruteForce': 'FTP-Patator'}
live2d['Label'] = live2d['Label'].replace(mapping)

In [122]:
live1d.Label.value_counts()

Label
PortScan       16983
BENIGN          8577
SSH-Patator      149
Name: count, dtype: int64

In [124]:
live2d.Label.value_counts()

Label
PortScan       75700
BENIGN          7633
FTP-Patator      454
SSH-Patator      110
Name: count, dtype: int64

In [ ]:
# =========================================================
# SPLIT X AND y
# =========================================================
X1 = live1d.drop('Label', axis=1)
y1 = live1d['Label']

X2 = live2d.drop('Label',axis=1)
y2 = live2d['Label']
# =========================================================
# CONVERT TO NUMERIC
# =========================================================

X1 = X1.apply(pd.to_numeric, errors='coerce')
X2 = X2.apply(pd.to_numeric, errors='coerce')
# =========================================================
# CHECK INF / NAN
# =========================================================

print("INF VALUES X1:", np.isinf(X1).sum().sum())
print("NAN VALUES X1:", np.isnan(X1).sum().sum())
print("INF VALUES X2:", np.isinf(X2).sum().sum())
print("NAN VALUES X2:", np.isnan(X2).sum().sum())
# =========================================================
# REPLACE INF
# =========================================================

X1 = X1.replace([np.inf, -np.inf], np.nan)
X2 = X2.replace([np.inf, -np.inf], np.nan)

# =========================================================
# FILL NAN
# =========================================================

X1 = X1.fillna(X1.median())
X2 = X2.fillna(X2.median())


# =========================================================
# VERIFY CLEAN DATA
# =========================================================

print("After preprocessing")
print("INF VALUES: X1 ", np.isinf(X1).sum().sum())
print("NAN VALUES: X1 ", np.isnan(X1).sum().sum())

print("After preprocessing")
print("INF VALUES: X2", np.isinf(X2).sum().sum())
print("NAN VALUES: X2", np.isnan(X2).sum().sum())

# =========================================================
# LABEL ENCODER
# =========================================================


y1 = le.transform(y1)
y2 = le.transform(y2)

# =========================================================
# VERIFY LABEL MAPPING
# =========================================================

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print("\nLabel Mapping:")
print(label_mapping)

INF VALUES X1: 379
NAN VALUES X1: 379
INF VALUES X2: 17
NAN VALUES X2: 7
After preprocessing
INF VALUES: X1  0
NAN VALUES: X1  0
After preprocessing
INF VALUES: X2 0
NAN VALUES: X2 0

Label Mapping:
{'BENIGN': 0, 'FTP-Patator': 1, 'PortScan': 2, 'SSH-Patator': 3}


In [126]:
from collections import Counter

c = Counter(y1)

print(c)

d = Counter(y2)

print(d)

Counter({2: 16983, 0: 8577, 3: 149})
Counter({2: 75700, 0: 7633, 1: 454, 3: 110})


In [130]:
y_pred = xg.predict(X1)

print("Accuracy:", accuracy_score(y1, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y1,
    y_pred))

Accuracy: 0.3385584814656346

Classification Report:

              precision    recall  f1-score   support

           0       0.34      1.00      0.50      8577
           2       0.00      0.00      0.00     16983
           3       1.00      0.88      0.94       149

    accuracy                           0.34     25709
   macro avg       0.45      0.63      0.48     25709
weighted avg       0.12      0.34      0.17     25709



Beningn and portscan confusion for xgboost

In [134]:
y_pred = xg.predict(X2)

print("Accuracy:", accuracy_score(y2, y_pred))

print("\nClassification Report:\n")
print(classification_report(
    y2,
    y_pred
    ))

Accuracy: 0.1054507312537993

Classification Report:

              precision    recall  f1-score   support

           0       0.09      1.00      0.17      7633
           1       1.00      0.69      0.81       454
           2       1.00      0.01      0.02     75700
           3       1.00      1.00      1.00       110

    accuracy                           0.11     83897
   macro avg       0.77      0.67      0.50     83897
weighted avg       0.91      0.11      0.04     83897



Model trained on CICIDS-17 is getting confused on Benign and portscan traffic on live capture. SSH and FTP Patator Decent.